# KUSIM vs ECLIPSE: a two-phase (oil/water) comparison

**Goal.** Run the layered-waterflood model of the bonus project (`course_plan/bonus_project.md`) in `kusim` **and** in ECLIPSE, and compare the results vector by vector. This notebook is *not* part of the course; it is a test bench.

**Workflow**

1. Section 1 defines the cases **once** (Python dictionaries, one per case, chosen with `CASE_NAME`). Both the `kusim` model and the ECLIPSE input deck are generated from the chosen case, so the two models cannot drift apart by accident. The folder of the case holds the deck and the ECLIPSE results; an existing deck is **compared** with the case and never overwritten.
2. Section 2 runs `kusim` and writes `output/kusim_summary.csv` (ECLIPSE vector names).
3. Section 3 writes `eclipse/KUSIM_BONUS.DATA`. **Run it in ECLIPSE.** The deck was written without access to ECLIPSE, so read it once and fix any keyword your version dislikes.
4. Section 4 loads the ECLIPSE summary (a CSV export, or the `.UNSMRY` file directly). If no ECLIPSE result is found the notebook still runs and shows `kusim` alone.
5. Sections 5 to 9 compare, test the time step, switch the known modelling differences on and off one at a time, and look into the negative pressures seen in long runs.

**What this notebook found.** A first comparison showed a large mismatch after water breakthrough that neither the time step nor the relative-permeability table explained. The cause was the upstream rule: the thesis and the FORTRAN take the mobility from the neighbour with the higher phase *pressure*, which is wrong in vertical flow. `kusim` now chooses the upstream block by *potential* (porting-notes decision 18); `Simulator(model, upstream="pressure")` gives the old behaviour, and Section 7 shows its effect. See `findings.md`.

**Rules of the comparison.** Compare *rates and pressures at the wells and the field*, and keep the pressure inside the PVT table range (400 to 5500 psia here). Outside it neither simulator is trustworthy (Section 8).

## 1. The case (single source of truth)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kusim import Simulator
from kusim.grid import Grid
from kusim.model import ReservoirModel, Equilibrium
from kusim.fluids import PVTTable, RockFluidTable
from kusim.wells import Producer, Injector

HERE = Path.cwd()                       # run Jupyter from the eclipse_comparison folder
OUT = HERE / "output";  OUT.mkdir(exist_ok=True)

C_KUSIM, C_ECL, C_DIFF = "#0072B2", "#D55E00", "#6b6b6b"     # colour-blind-safe pair; the line style differs too
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 100, "legend.frameon": False})

SWOF = [[0.20, 0.00, 0.85, 8.0],         # Sw, krw, kro, Pcow (psi): the same table in both cases
        [0.40, 0.05, 0.35, 2.5],
        [0.60, 0.20, 0.10, 0.8],
        [0.75, 0.40, 0.02, 0.3],
        [0.85, 0.60, 0.00, 0.15],
        [1.00, 1.00, 0.00, 0.0]]

CASES = {
    # the case that the ECLIPSE results in eclipse/ belong to (5 x 5 x 3 blocks, 2000 days)
    "deck_5x5x3": dict(
        nx=5, ny=5, nz=3, dx=500.0, dy=500.0, dz=50.0, top=8000.0,
        kx=[200.0, 1000.0, 200.0],           # md, one value per layer (top first)
        ky_over_kx=0.8, kz_over_kx=0.1, phi=0.20, cr=4e-6,
        datum_depth=8075.0, datum_pressure=4500.0, woc=8500.0,     # capillary-gravity equilibrium
        rho_o=49.0, rho_w=63.0,              # lb/ft3 at surface
        swc=0.20, sor=0.15, swof=SWOF,
        pvdo=[[300.0, 1.25, 1.0], [800.0, 1.20, 1.1], [6000.0, 1.15, 2.0]],   # p (psia), Bo (rb/STB), mu_o (cp)
        bw=1.02, mu_w=0.8,                   # constant water PVT
        prod=dict(name="P1", i=1, j=1, mode="liquid_rate", rate=10000.0, min_bhp=2000.0, radius=0.35),
        inj=dict(name="I1", i=5, j=5, mode="water_rate", rate=11000.0, radius=0.35),
        t_end=2000.0),
    # the bonus-project model (course_plan/bonus_project.md; 6 x 4 x 3 blocks, 1200 days)
    "bonus_6x4x3": dict(
        nx=6, ny=4, nz=3, dx=400.0, dy=400.0, dz=40.0, top=7500.0,
        kx=[120.0, 600.0, 250.0],
        ky_over_kx=0.8, kz_over_kx=0.1, phi=0.22, cr=5e-6,
        datum_depth=7580.0, datum_pressure=4200.0, woc=8000.0,
        rho_o=50.0, rho_w=64.0,
        swc=0.20, sor=0.15, swof=SWOF,
        pvdo=[[400.0, 1.30, 0.9], [1000.0, 1.24, 1.0], [5500.0, 1.17, 1.8]],   # bubble point 400 psia
        bw=1.01, mu_w=0.7,
        prod=dict(name="P1", i=1, j=1, mode="liquid_rate", rate=8000.0, min_bhp=1500.0, radius=0.35),
        inj=dict(name="I1", i=6, j=4, mode="water_rate", rate=9000.0, radius=0.35),
        t_end=1200.0),
}
ECLIPSE_FOLDER = {"deck_5x5x3": HERE / "eclipse", "bonus_6x4x3": HERE / "eclipse_runs" / "bonus_case"}

CASE_NAME = "deck_5x5x3"        # <- choose the case. Well names must stay P1 / I1: the vectors below use them
CASE = CASES[CASE_NAME]
ECL = ECLIPSE_FOLDER[CASE_NAME]; ECL.mkdir(parents=True, exist_ok=True)
PVT_RANGE = (CASE["pvdo"][0][0], CASE["pvdo"][-1][0])
print("case:", CASE_NAME, "| ECLIPSE folder:", ECL, "| PVT table range (psia):", PVT_RANGE)

### Building the `kusim` model

Two switches are provided for the *diagnostic* runs of Section 7 (the defaults are the plain `kusim` behaviour):

* `eclipse_pvt=True`: I believe ECLIPSE interpolates `1/B` and `1/(B*mu)` linearly in a `PVDO` table, whereas `kusim` interpolates `1/B` and `mu` linearly. This switch resamples the table on a fine grid with the ECLIPSE rule. (Check the `PVDO` entry of the ECLIPSE Technical Description for your version before relying on it.)
* `wellbore_head=False`: switch off the wellbore pressure head that `kusim` adds between the perforations of a multi-layer well.

`run_kusim(..., upstream="pressure")` (a simulator option, not a model one) gives the thesis rule for the upstream block instead of the default, which is by potential.

In [ ]:
def pvt_table(case, eclipse_style=False):
    p, bo, mu = np.array(case["pvdo"]).T
    if not eclipse_style:
        return PVTTable(p, bo, [case["bw"]] * len(p), mu, [case["mu_w"]] * len(p))
    pf = np.linspace(p[0], p[-1], 511)                       # 10 psi spacing, contains the table nodes
    inv_b = np.interp(pf, p, 1.0 / bo)
    inv_bmu = np.interp(pf, p, 1.0 / (bo * mu))
    return PVTTable(pf, 1.0 / inv_b, [case["bw"]] * len(pf), inv_b / inv_bmu, [case["mu_w"]] * len(pf))


def build_kusim(case, dt=5.0, t_end=None, eclipse_pvt=False, wellbore_head=True, min_bhp="case",
                producer_rate=None, injector=True):
    nx, ny, nz = case["nx"], case["ny"], case["nz"]
    g = Grid([case["dx"]] * nx, [case["dy"]] * ny, [case["dz"]] * nz, top_depth=case["top"])
    kx = np.repeat(case["kx"], nx * ny)
    rf = RockFluidTable(*np.array(case["swof"])[:, [0, 2, 1, 3]].T)        # sw, kro, krw, pcow
    gamma_o, gamma_w = case["rho_o"] / 62.4, case["rho_w"] / 62.4
    datum_p0 = case["datum_pressure"] - 0.433 * gamma_o * case["datum_depth"]   # pressure at depth 0
    p, w = case["prod"], case["inj"]
    gravity = None if wellbore_head else 0.0
    injectors = [Injector(g.column_blocks(w["i"], w["j"]), w["mode"], w["rate"], radius=w["radius"],
                          wellbore_gravity=gravity)] if injector else []
    return ReservoirModel(
        grid=g, kx=kx, ky=kx * case["ky_over_kx"], kz=kx * case["kz_over_kx"],
        porosity=case["phi"], rock_compressibility=case["cr"],
        pvt=pvt_table(case, eclipse_pvt), rock_fluid=rf,
        gamma_o=gamma_o, gamma_w=gamma_w, swc=case["swc"], sor=case["sor"],
        initial=Equilibrium(datum_pressure=datum_p0, woc_depth=case["woc"]),
        dt=dt, t_end=t_end or case["t_end"],
        producers=[Producer(g.column_blocks(p["i"], p["j"]), p["mode"], producer_rate or p["rate"],
                            radius=p["radius"], min_bhp=p["min_bhp"] if min_bhp == "case" else min_bhp,
                            wellbore_gravity=gravity)],
        injectors=injectors,
    )

## 2. Run `kusim`

`summary` uses ECLIPSE vector names: `WBHP:P1`, `WOPR:P1`, `WWCT:P1`, `WWIR:I1`, `FPRP` (pore-volume-weighted average pressure), `FOPT`, ... Rates are those of the step that ends at `TIME`, like ECLIPSE report steps.

In [ ]:
def run_kusim(case, dt=5.0, upstream="potential", **kw):
    sim = Simulator(build_kusim(case, dt, **kw), upstream=upstream)
    res = sim.run()
    pv = sim.grid.volume * sim.model.porosity
    w = res.wells
    P = w[w.well == "P1"].reset_index(drop=True)
    I = w[w.well == "I1"].reset_index(drop=True)
    fprp = (res.pressure[1:] * pv).sum(axis=1) / pv.sum()        # one row per step (snapshot 0 is t = 0)
    summary = pd.DataFrame({
        "TIME": P.time, "FPRP": fprp,
        "WBHP:P1": P.bhp, "WBHP:I1": I.bhp,
        "WOPR:P1": P.oil_rate, "WWPR:P1": P.water_rate, "WWCT:P1": P.water_cut, "WWIR:I1": I.water_rate,
        "FOPT": P.cum_oil, "FWPT": P.cum_water, "FWIT": I.cum_water,
    })
    return summary, res, sim

kus, kus_res, kus_sim = run_kusim(CASE, dt=5.0)
kus.to_csv(OUT / "kusim_summary.csv", index=False)
print("stop reason:", kus_res.stop_reason)
kus.iloc[[0, 1, 39, 99, 199, -1]].round(3)

In [ ]:
def check_pvt_range(p_all, label="", lo=PVT_RANGE[0], hi=PVT_RANGE[1]):
    """Say how much of the run left the PVT table (both simulators are unreliable out there)."""
    p_all = np.asarray(p_all)
    below, above = (p_all < lo).mean() * 100, (p_all > hi).mean() * 100
    flag = "OK     " if below + above == 0 else "WARNING"
    print(f"{flag} {label}: pressure {p_all.min():.0f} .. {p_all.max():.0f} psia; "
          f"{below:.1f}% of values below {lo:.0f}, {above:.1f}% above {hi:.0f}")

check_pvt_range(kus_res.pressure, "kusim base case, all cells")

## 3. The ECLIPSE deck

`eclipse_deck()` writes an ECLIPSE 100 deck (FIELD units, `OIL` + `WATER`) from the same `CASE`. The deck goes to the folder of the case **only if there is none**; if there is one it is compared with the case (a mismatch is reported, the file is never changed). Mapping, line by line:

| `kusim` | ECLIPSE |
|---|---|
| `Grid`, `top_depth`, `kx/ky/kz` per layer | `DIMENS`, `DX/DY/DZ`, `TOPS`, `PERMX/PERMY/PERMZ` |
| `Equilibrium(datum_pressure, woc_depth)` | `EQUIL` (datum depth, pressure, WOC depth, Pcow at WOC = 0) |
| rel-perm + Pcow table | `SWOF` (same columns: Sw, krw, kro, Pcow) |
| `PVTTable` oil columns | `PVDO`; water: `PVTW` with zero compressibility and viscosibility |
| `rock_compressibility` | `ROCK` (reference pressure = initial pressure) |
| oil/water gravity = rho/62.4 | `DENSITY` (oil, water, gas unused) |
| `Producer(mode="liquid_rate", min_bhp)` | `WCONPROD ... LRAT ... BHP` (switches to BHP control at the limit) |
| `Injector(mode="water_rate")` | `WCONINJE ... RATE` (BHP limit set high so it never acts, as in `kusim`) |
| well radius 0.35 ft, skin 0 | `COMPDAT` diameter 0.70 ft |
| BHP is the pressure at the shallowest perforation | `WELSPECS` reference depth = centre of the top layer |
| `dt = 5` days | `TSTEP` of 5-day report steps, `TUNING` max step 5 days |
| `FPRP` from the results | `FPRP` (**pore-volume** average; ECLIPSE `FPR` is hydrocarbon-pore-volume weighted and is *not* comparable) |

In [ ]:
def eclipse_deck(case, report_step=5.0):
    nx, ny, nz, nxy = case["nx"], case["ny"], case["nz"], case["nx"] * case["ny"]
    n = nxy * nz
    p, w = case["prod"], case["inj"]
    assert (p["mode"], w["mode"]) == ("liquid_rate", "water_rate"), "the deck writer only knows LRAT / RATE wells"
    layers = lambda vals: " ".join(f"{nxy}*{v:g}" for v in vals)
    rows = lambda table: "\n".join(" ".join(f"{v:g}" for v in r) for r in table)
    kx = case["kx"]
    z_ref = case["top"] + case["dz"] / 2                        # centre of the top layer
    return f"""-- KUSIM comparison deck, written by kusim_vs_eclipse.ipynb (not tested in ECLIPSE: check keywords)
RUNSPEC
TITLE
 KUSIM bonus case: layered waterflood
DIMENS
 {nx} {ny} {nz} /
OIL
WATER
FIELD
TABDIMS
 1 1 {max(20, len(case["swof"]))} 20 /
EQLDIMS
 1 /
WELLDIMS
 2 {nz} 1 2 /
START
 1 JAN 2000 /
UNIFOUT

GRID
INIT
DX
 {n}*{case["dx"]:g} /
DY
 {n}*{case["dy"]:g} /
DZ
 {n}*{case["dz"]:g} /
TOPS
 {nxy}*{case["top"]:g} /
PORO
 {n}*{case["phi"]:g} /
PERMX
 {layers(kx)} /
PERMY
 {layers([k * case["ky_over_kx"] for k in kx])} /
PERMZ
 {layers([k * case["kz_over_kx"] for k in kx])} /

PROPS
SWOF
{rows(case["swof"])} /
DENSITY
 {case["rho_o"]:g} {case["rho_w"]:g} 0.0702 /
PVTW
 {case["datum_pressure"]:g} {case["bw"]:g} 0.0 {case["mu_w"]:g} 0.0 /
ROCK
 {case["datum_pressure"]:g} {case["cr"]:g} /
PVDO
{rows(case["pvdo"])} /

SOLUTION
EQUIL
 {case["datum_depth"]:g} {case["datum_pressure"]:g} {case["woc"]:g} 0.0 /
RPTRST
 BASIC=1 /

SUMMARY
FPRP
FOPT
FWPT
FWIT
FOPR
FWPR
FWIR
FWCT
WBHP
 {p["name"]} {w["name"]} /
WOPR
 {p["name"]} /
WWPR
 {p["name"]} /
WWCT
 {p["name"]} /
WWIR
 {w["name"]} /

SCHEDULE
TUNING
 1 {report_step:g} /
/
/
WELSPECS
 {p["name"]} G1 {p["i"]} {p["j"]} {z_ref:g} OIL /
 {w["name"]} G1 {w["i"]} {w["j"]} {z_ref:g} WATER /
/
COMPDAT
 {p["name"]} {p["i"]} {p["j"]} 1 {nz} OPEN 1* 1* {2 * p["radius"]:g} /
 {w["name"]} {w["i"]} {w["j"]} 1 {nz} OPEN 1* 1* {2 * w["radius"]:g} /
/
WCONPROD
 {p["name"]} OPEN LRAT 3* {p["rate"]:g} 1* {p["min_bhp"]:g} /
/
WCONINJE
 {w["name"]} WATER OPEN RATE {w["rate"]:g} 1* 9999 /
/
TSTEP
 {round(case["t_end"] / report_step)}*{report_step:g} /
END
"""

def check_deck(case, folder, name="KUSIM_BONUS.DATA"):
    """Write the deck if the folder has none; otherwise compare it with the case and never overwrite it."""
    import re
    path = folder / name
    new = eclipse_deck(case)
    if not path.exists():
        path.write_text(new)
        print("written:", path)
        return
    tokens = lambda text: [t for t in re.sub(r"--.*", "", text).split() if t.upper() != "RSM"]
    if tokens(path.read_text()) == tokens(new):
        print("OK: the deck", path, "matches CASE", repr(CASE_NAME))
    else:
        print("WARNING: the deck", path, "is DIFFERENT from CASE", repr(CASE_NAME), ": the ECLIPSE results in this "
              "folder may belong to another case (the file was not changed). Write the new deck to another folder to compare.")

check_deck(CASE, ECL)
print(eclipse_deck(CASE)[:700], "...")

## 4. Load the ECLIPSE results

Put the result in the `eclipse/` folder, using one of these:

* **`KUSIM_BONUS.csv`**: a CSV export (ResInsight, Petrel, `res2df`, ...) with a time column in **days** (`TIME`, `DAYS` or `Time`) and one column per vector named `WBHP:P1`, `WOPR:P1`, `FPRP`, ... If your export names the columns differently, edit `COLUMN_MAP` below.
* **`KUSIM_BONUS.UNSMRY`** (with its `.SMSPEC`): read directly if the optional package `resdata` (or the older `ecl`) is installed: `pip install resdata`.

Nothing found: the notebook continues with `kusim` only.

In [ ]:
COLUMN_MAP = {}          # e.g. {"Well BHP:P1": "WBHP:P1", "Days": "TIME"}
VECTORS = ["FPRP", "WBHP:P1", "WBHP:I1", "WOPR:P1", "WWPR:P1", "WWCT:P1", "WWIR:I1", "FOPT", "FWPT", "FWIT"]

def load_eclipse(folder, stem="KUSIM_BONUS"):
    csv = next((f for f in folder.glob(stem + ".*") if f.suffix.lower() == ".csv"), None)
    if csv is not None:
        df = pd.read_csv(csv).rename(columns=COLUMN_MAP)
        df.columns = ["TIME" if c.strip().upper() in ("TIME", "DAYS") else c.strip().upper().replace(" ", ":")
                      for c in df.columns]
        return df.sort_values("TIME").reset_index(drop=True)
    unsmry = next((f for f in folder.glob(stem + ".*") if f.suffix.upper() in (".UNSMRY", ".SMSPEC")), None)
    if unsmry is not None:
        try:
            from resdata.summary import Summary as _Sum
        except ImportError:
            try:
                from ecl.summary import EclSum as _Sum
            except ImportError:
                print("Found", unsmry.name, "but neither `resdata` nor `ecl` is installed (pip install resdata).")
                return None
        s = _Sum(str(folder / stem))
        data = {"TIME": np.asarray(s.numpy_vector("TIME"))}
        for v in VECTORS:
            if v in s:
                data[v] = np.asarray(s.numpy_vector(v))
        return pd.DataFrame(data)
    return None

ecl = load_eclipse(ECL)
if ecl is None:
    print("No ECLIPSE results found in", ECL, "-> plots will show kusim only.")
else:
    ecl = ecl[ecl.TIME > 0].reset_index(drop=True)
    print("ECLIPSE vectors loaded:", [c for c in ecl.columns if c != "TIME"])
    print("missing (will be skipped):", [v for v in VECTORS if v not in ecl.columns])
    print(f"ECLIPSE time range: {ecl.TIME.min():g} .. {ecl.TIME.max():g} days, {len(ecl)} rows")

## 5. Compare

Top panel: both curves (`kusim` solid blue, ECLIPSE dashed orange). Bottom panel: `kusim - ECLIPSE`, with ECLIPSE interpolated onto the `kusim` times. Look at **shapes and timing** first (when does the water cut start, where is the pressure minimum), then at the size of the difference.

In [ ]:
LABELS = {"FPRP": "Field average pressure (psia)", "WBHP:P1": "Producer BHP (psia)", "WBHP:I1": "Injector BHP (psia)",
          "WOPR:P1": "Oil rate (STB/d)", "WWPR:P1": "Water production rate (STB/d)", "WWCT:P1": "Water cut (fraction)",
          "WWIR:I1": "Water injection rate (STB/d)", "FOPT": "Cumulative oil (STB)",
          "FWPT": "Cumulative water produced (STB)", "FWIT": "Cumulative water injected (STB)"}

def compare_plot(vec, kus, ecl=None, xlim=None, extra=None, title=None):
    """Overlay + difference. ``extra`` = {label: summary_df} adds more kusim runs (thin lines)."""
    has_ecl = ecl is not None and vec in ecl.columns
    fig, (ax, ad) = plt.subplots(2, 1, figsize=(7.5, 5.2), sharex=True, gridspec_kw=dict(height_ratios=[3, 1]))
    for lab, df in (extra or {}).items():
        ax.plot(df.TIME, df[vec], lw=1.2, label=lab)
    ax.plot(kus.TIME, kus[vec], color=C_KUSIM, lw=2, label="kusim")
    if has_ecl:
        ax.plot(ecl.TIME, ecl[vec], color=C_ECL, lw=2, ls="--", label="ECLIPSE")
        diff = kus[vec] - np.interp(kus.TIME, ecl.TIME, ecl[vec])
        ad.plot(kus.TIME, diff, color=C_DIFF, lw=1.5)
        ad.axhline(0, color="k", lw=0.6)
    else:
        ad.text(0.5, 0.5, "no ECLIPSE result loaded", transform=ad.transAxes, ha="center", va="center", color=C_DIFF)
    ax.set_ylabel(LABELS[vec]); ad.set_ylabel("kusim - ECL"); ad.set_xlabel("time (days)")
    ax.set_title(title or LABELS[vec], loc="left", fontsize=11)
    if xlim: ax.set_xlim(*xlim)
    ax.legend()
    fig.tight_layout(); plt.show()

for v in ["FPRP", "WBHP:P1", "WBHP:I1", "WOPR:P1", "WWCT:P1", "WWIR:I1"]:
    compare_plot(v, kus, ecl)

In [ ]:
# The first days: where the start-up transient (compressibility, well index) is tested.
compare_plot("WBHP:P1", kus, ecl, xlim=(0, 60), title="Producer BHP, first 60 days")
compare_plot("WBHP:I1", kus, ecl, xlim=(0, 60), title="Injector BHP, first 60 days")

In [ ]:
def comparison_table(kus, ecl, vectors=VECTORS, times=(5, 10, 100, 300, 600, 1200, 2000)):
    times = [t for t in times if t <= kus.TIME.max()]
    if ecl is None:
        return kus.set_index("TIME").loc[[t for t in times if t in set(kus.TIME)], vectors].round(3)
    rows = []
    for v in vectors:
        if v not in ecl.columns:
            continue
        e_on_k = np.interp(kus.TIME, ecl.TIME, ecl[v])
        d = kus[v].to_numpy() - e_on_k
        scale = np.nanmean(np.abs(e_on_k)) or np.nan
        row = {"vector": v, "mean |diff| %": 100 * np.nanmean(np.abs(d)) / scale, "max |diff|": np.nanmax(np.abs(d))}
        for t in times:
            i = int((kus.TIME - t).abs().idxmin())
            row[f"diff @{t:g}d"] = d[i]
        rows.append(row)
    return pd.DataFrame(rows).set_index("vector").round(3)

comparison_table(kus, ecl)

### Cumulative volumes

Cumulative oil produced and water injected are the integral of the rates. If `FOPT` or `FWIT` disagree by more than the integral of the rate differences, the two runs are not solving the same problem (Section 8 checks the `kusim` material balance cell by cell).

In [ ]:
for v in ["FOPT", "FWPT", "FWIT"]:
    compare_plot(v, kus, ecl)

## 6. Time step

`kusim` is IMPES: the explicit saturation update makes the answer depend on `dt`, and the error of a large step can look like a "simulator difference". ECLIPSE chooses its own steps (limited to 5 days here by `TUNING`). Run `kusim` at several steps and see which one the ECLIPSE curve resembles. If the `kusim` curves for `dt = 1` and `dt = 2.5` agree with each other, the remaining gap to ECLIPSE is *not* the time step.

In [ ]:
runs = {f"kusim dt={dt:g}": run_kusim(CASE, dt=dt)[0] for dt in (1.0, 2.5, 10.0)}
for v in ["WOPR:P1", "WWCT:P1", "WBHP:P1"]:
    compare_plot(v, kus, ecl, extra=runs, title=LABELS[v] + ": time-step study")

## 7. Known modelling differences: switch them one at a time

These are the differences I know of between the two models. For each variant the plot shows the effect, so you can see which one explains a gap.

| # | Difference | Direction of the effect |
|---|---|---|
| 0 | **Upstream rule** (resolved): the thesis and the FORTRAN choose the upstream block by phase *pressure*; with gravity that is wrong in vertical flow and keeps sinking water from moving down. `kusim` now uses the *potential* (decision 18); the variant `upstream="pressure"` brings the old behaviour back | **large**: after water breakthrough the pressures and rates of the thesis rule are far from ECLIPSE; with the potential rule the RMS error of the pressures falls by about an order of magnitude (`findings.md`) |
| 1 | **Wellbore head** in the multi-layer wells: `kusim` uses oil gravity (producer) / water gravity (injector); ECLIPSE uses the density of the flowing mixture | BHP shift of a few psi, at most 80 ft x (64-50)/144 = 8 psi |
| 2 | **PVT interpolation** (see Section 1) | small shift in mobility between table nodes |
| 3 | `kusim` clips `Sw` to `[swc, 1 - sor]`; ECLIPSE does not | mass is not conserved when a block would exceed `1 - sor` |
| 4 | Pressure gradient: `0.433 * rho/62.4` (kusim) vs `rho / 144` (ECLIPSE) | about 0.1 % of the gravity head: negligible |
| 5 | Producer limit check: `kusim` tests `min_bhp` at the **end** of the step (the step that violates the limit is still run at the old rate); ECLIPSE enforces it inside the step | only matters if the BHP limit is reached |
| 6 | Explicit saturation (IMPES) vs fully implicit | Section 6 |

Differences 0 to 2 have switches below; 3 to 6 do not.

In [ ]:
variants = {
    "kusim, thesis upstream (pressure)": dict(upstream="pressure"),
    "kusim, no wellbore head": dict(wellbore_head=False),
    "kusim, ECLIPSE-style PVT": dict(eclipse_pvt=True),
    "kusim, both": dict(wellbore_head=False, eclipse_pvt=True),
}
var_runs = {lab: run_kusim(CASE, dt=5.0, **kw)[0] for lab, kw in variants.items()}
for v in ["WBHP:P1", "WBHP:I1", "FPRP", "WWCT:P1"]:
    compare_plot(v, kus, ecl, extra=var_runs, title=LABELS[v] + ": effect of modelling switches")

In [ ]:
if ecl is not None:
    check = [v for v in ["FPRP", "WBHP:P1", "WBHP:I1", "WOPR:P1", "WWCT:P1"] if v in ecl.columns]
    allruns = {"kusim (base)": kus, **var_runs}
    table = {lab: {v: np.sqrt(np.mean((df[v] - np.interp(df.TIME, ecl.TIME, ecl[v])) ** 2)) for v in check}
             for lab, df in allruns.items()}
    print("RMS difference against ECLIPSE (smaller is closer):")
    display(pd.DataFrame(table).T.round(3))
else:
    print("Load ECLIPSE results (Section 4) to rank the variants.")

## 8. The negative pressure in long runs

This section always uses the **bonus-project data** (`BONUS`), whatever `CASE_NAME` is.

**What was seen:** run `kusim` for a long time with the producer pulling more fluid than the reservoir can replace and the pressure goes through zero and keeps falling. Two explanations were on the table: (a) no gas phase, so nothing supports the pressure below the bubble point, or (b) a genuine error in the simulator. The tests below separate them. In the bonus data the bubble point is **400 psia**, the lowest pressure of the PVT table.

**How mass conservation is checked.** After every step the change of oil and water in place is compared with the volumes that the wells produced and injected in that step, and the errors are added up (`step_balance` below; the pore volume follows the rock compressibility from the pressure at the *start of the step*, which is what the scheme assumes). An earlier version of this notebook compared the in-place volumes at the end of the run with those at the start, with the pore volume referred to the initial pressure. That is path-dependent and overstated the error by an order of magnitude; do not use it.

**Findings (the cells below repeat them):**

1. **Inside the PVT table range the scheme conserves mass.** In the base case (pressure 2400 to 4900 psia) the errors summed over all steps are 0.005 % of the initial oil in place and 0.014 % of the initial water in place; a depletion run that stays above 1500 psia gives less than 0.001 %. (With the old thesis rule for the upstream block the base-case errors were ten times larger, 0.05 % and 0.2 %.)
2. **The negative pressure needs a producer without a BHP limit.** With `min_bhp=1500` the producer switches to BHP control and the field pressure settles at about 1514 psia. Remove the limit (`min_bhp=None`) in a closed box and the rate-controlled well keeps withdrawing: the pressure falls linearly (a few psi per day) through zero and beyond. In ECLIPSE the well would hit its minimum BHP and change control.
3. **Hypothesis (a) is right about the physics.** The model is dead oil plus water. Real oil below the bubble point releases gas, whose compressibility holds the pressure up. This model cannot do that, so any result below 400 psia is outside what it represents. It does not by itself explain the *negative* value; item 2 does.
4. **There is also a numerical inconsistency outside the table.** `Table.lookup` is *flat* beyond the ends (1/Bo constant) but `Table.slope` returns the slope of the *end segment*, so the storage coefficient `d(1/Bo)/dP` stays non-zero. The pressure equation then believes each psi of drop releases oil that the fluid properties never account for, and the material balance stops closing: **about 1.5 % of the oil in place is missing after 600 days** in run C below. The docstring in `tables.py` says this follows the original program. Extending the PVT table down to 14.7 psia does *not* cure the negative pressure (checked); item 2 remains the cause.

**Conclusion.** Not a fundamental error in the in-range scheme, but two real limits: no gas below the bubble point, and an unbounded rate-controlled well. Keep `min_bhp` above the bubble point in every comparison with ECLIPSE. Possible `kusim` improvements (not made, they change the simulator): warn when the pressure leaves the PVT range, and make the slope zero outside the table so that the properties stay consistent (listed in `docs/porting-notes.md`, "Not done / open").

In [ ]:
BONUS = CASES["bonus_6x4x3"]

def step_balance(label, case, dt=5.0, t_end=1200.0, **kw):
    """Oil and water balance of every step, added up (uses a private attribute of the simulator, for testing only)."""
    sim = Simulator(build_kusim(case, dt, t_end=t_end, **kw))
    model, vol = sim.model, sim.grid.volume
    def in_place(p, s, p_ref):
        pv = vol * model.porosity * (1 + model.rock_compressibility * (p - p_ref))
        return ((pv * (1 - s) * model.pvt.inv_bo(p)).sum() / 5.615,
                (pv * s * model.pvt.inv_bw(p - model.rock_fluid.pcow_at(s))).sum() / 5.615)
    oil0, water0 = in_place(sim.p_o, sim.s_w, sim.p_o)
    err_oil = err_water = 0.0
    while not sim.finished:
        p_old, s_old = sim.p_o.copy(), sim.s_w.copy()
        oil_b, water_b = in_place(p_old, s_old, p_old)
        sim.step()
        oil_a, water_a = in_place(sim.p_o, sim.s_w, p_old)
        wells = sim._step_wells
        produced_oil, produced_water = wells["P1"]["q_o"].sum() * dt, wells["P1"]["q_w"].sum() * dt
        injected = wells["I1"]["q_w"].sum() * dt if "I1" in wells else 0.0
        err_oil += (oil_b - oil_a) - produced_oil
        err_water += (water_a - water_b) - (injected - produced_water)
    res = sim.results()
    print(f"{label}\n  mean pressure {res.pressure[0].mean():.0f} -> {res.pressure[-1].mean():.0f} psia "
          f"(min {res.pressure.min():.0f}, max {res.pressure.max():.0f})"
          f"\n  oil error added over the steps:   {100 * err_oil / oil0:+.3f} % of initial oil in place"
          f"\n  water error added over the steps: {100 * err_water / water0:+.3f} % of initial water in place\n")
    return res

step_balance("A. base case, 1200 d (inside the PVT range)", BONUS)
step_balance("B. depletion, 3000 STB/d, floor 1500 psia, 200 d", BONUS, t_end=200, producer_rate=3000.0, injector=False)
step_balance("C. depletion, 2500 STB/d, NO pressure floor, 600 d (leaves the table)", BONUS,
             t_end=600, producer_rate=2500.0, injector=False, min_bhp=None)

In [ ]:
long = Simulator(build_kusim(BONUS, 5.0, t_end=2000.0, producer_rate=2500.0, injector=False, min_bhp=None)).run()
bubble_point = BONUS["pvdo"][0][0]
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(long.times, long.pressure.mean(axis=1), color=C_KUSIM, lw=2, label="mean block pressure, no producer BHP limit")
ax.axhline(bubble_point, color=C_DIFF, ls=":", label="bubble point / start of PVT table")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("time (days)"); ax.set_ylabel("psia")
ax.set_title("Closed box, rate-controlled producer, no BHP limit", loc="left", fontsize=11)
ax.legend(); fig.tight_layout(); plt.show()

## 9. If the results do not agree: a checklist

0. Which **upstream rule**? The default is by potential (right with gravity). `upstream="pressure"` is the thesis rule and fails in vertical flow (Section 7).
1. Same **initial state**? Compare `FPRP` at the first report step. Check `PVTW` (reference pressure, `Bw`) and `ROCK`.
2. Same **well model**? Look at the first steps of `WBHP:P1` and `WBHP:I1` (Section 5, 60-day zoom). A constant offset points to the well index or the wellbore head (Section 7). Check the `COMPDAT` diameter (0.70 ft), skin and the reference depth.
3. Same **volumes**? `FOPT` and `FWIT` should be the integral of the same rates.
4. **Water breakthrough time** (`WWCT:P1`) shifted? Look at the relative-permeability end points and at Section 6 (numerical dispersion of the explicit scheme).
5. Is any pressure **outside 400 to 5500 psia**? ECLIPSE extrapolates `PVDO` linearly and `kusim` is flat (Section 8).
6. Only then look at cell-by-cell differences: both codes number the cells with *i* fastest, then *j*, then *k* from the top, so an ECLIPSE restart array (`PRESSURE`, `SWAT`) can be compared directly with `kus_res.pressure[-1]` and `kus_res.sw[-1]`.

In [ ]:
# Optional: cell-by-cell comparison at the last report step. Fill in two arrays of length 72 from your
# ECLIPSE restart file (same cell order as kusim) and run this cell.
ecl_pressure_last = None     # e.g. np.loadtxt("eclipse/PRESSURE_1200.txt")
ecl_swat_last = None
if ecl_pressure_last is not None and ecl_swat_last is not None:
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for ax, (name, a, b) in zip(axes, [("pressure (psia)", kus_res.pressure[-1], ecl_pressure_last),
                                       ("Sw", kus_res.sw[-1], ecl_swat_last)]):
        ax.scatter(b, a, s=18, color=C_KUSIM)
        lim = [min(a.min(), b.min()), max(a.max(), b.max())]
        ax.plot(lim, lim, color=C_DIFF, lw=1)
        ax.set_xlabel("ECLIPSE"); ax.set_ylabel("kusim"); ax.set_title(name, loc="left")
    fig.tight_layout(); plt.show()
else:
    print("No cell arrays supplied (optional).")